# 01_data_generation

In diesem Notebook werden synthetische Daten für ein Credit-Risk- und Schufa-Score-Projekt erzeugt.

## Ziel
Das Ziel ist die Simulation eines Datensatzes, mit dem folgende Fragestellungen analysiert werden können:

- Unterschiede zwischen einem alten und einem neuen Score-System
- Zusammenhang zwischen Einkommen, Liquidität und Kreditvergabe
- Einfluss negativer Einträge auf die Kreditentscheidung
- Vergleich zwischen Score-basierter Ablehnung und tatsächlichem Ausfallrisiko

## Hinweis
Alle Daten in diesem Notebook sind synthetisch erzeugt und stellen keine echten SCHUFA-Daten dar.
Die verwendeten Score-Logiken sind projektinterne Approximationen und keine echten Modelle realer Auskunftdatenbanken oder Banken.

## Data Dictionary

- age: Alter der Person
- net_income: monatliches Nettoeinkommen
- employment_status: Beschäftigungsstatus
- marital_status: Familienstand
- dependents: Anzahl unterhaltspflichtiger Personen
- monthly_expenses: geschätzte monatliche Ausgaben
- existing_loans: Anzahl bestehender Kredite
- credit_amount: beantragter Kreditbetrag
- debt_to_income: Verhältnis Kreditbetrag zu Jahreseinkommen
- late_payments: Anzahl verspäteter Zahlungen
- hard_inquiries: Anzahl Kreditanfragen
- credit_history_years: Länge der Kredithistorie
- negative_entry: negativer Bonitätseintrag
- collection_cases: Inkassofälle
- bank_balance: Kontostand / Liquiditätsindikator
- liquidity_ratio: Verhältnis Kontostand zu monatlichen Ausgaben
- old_score: synthetischer alter Score
- new_score: synthetischer neuer Score
- credit_approved_old_system: Kreditentscheidung im alten System
- credit_approved_new_system: Kreditentscheidung im neuen System
- default: simuliertes Ausfallereignis

In [44]:
import numpy as np
import pandas as pd
import random
import os

np.random.seed(42)
random.seed(42)

print("Imports erfolgreich geladen.")

Imports erfolgreich geladen.


In [45]:
from google.colab import files

uploaded = files.upload()

Saving test_credit_upload_file.csv to test_credit_upload_file.csv


In [47]:
# Zeigt an, welche Dateien hochgeladen wurden
uploaded.keys()

dict_keys(['test_credit_upload_file.csv'])

In [49]:
import pandas as pd

df_uploaded = pd.read_csv("test_credit_upload_file.csv")
df_uploaded.head()

,age,net_income,employment_status,existing_loans,credit_amount,late_payments,negative_entry
0,25,1800,teilzeit,0,2000,0,0
1,34,3200,angestellt,1,8000,1,0
2,45,4100,selbstständig,2,12000,0,1
3,29,2400,angestellt,0,3500,2,0
4,52,5200,selbstständig,3,20000,1,1


## Parameter

In [50]:
N = 10000
print("Anzahl zu generierender Datensätze:", N)

Anzahl zu generierender Datensätze: 10000


## Soziodemografische Merkmale

In [51]:
age = np.random.randint(18, 70, N)

employment_status = np.random.choice(
    ['angestellt', 'selbstständig', 'arbeitslos', 'teilzeit'],
    size=N,
    p=[0.58, 0.17, 0.08, 0.17]
)

marital_status = np.random.choice(
    ['ledig', 'verheiratet', 'geschieden'],
    size=N,
    p=[0.45, 0.40, 0.15]
)

dependents = np.random.choice(
    [0, 1, 2, 3, 4],
    size=N,
    p=[0.45, 0.22, 0.18, 0.10, 0.05]
)

## Einkommen abhängig vom Beschäftigungsstatus

In [52]:
net_income = []

for status in employment_status:
    if status == 'angestellt':
        income = np.random.normal(3200, 900)
    elif status == 'selbstständig':
        income = np.random.normal(3500, 1400)
    elif status == 'teilzeit':
        income = np.random.normal(1700, 500)
    else:  # arbeitslos
        income = np.random.normal(1100, 300)
    net_income.append(income)

net_income = np.array(net_income)
net_income = np.clip(net_income, 800, 12000)

## Finanzielle Merkmale

In [53]:
monthly_expenses = (
    net_income * np.random.uniform(0.45, 0.75, N)
    + dependents * np.random.uniform(150, 350, N)
)
monthly_expenses = np.clip(monthly_expenses, 500, None)

existing_loans = np.random.poisson(lam=1.2, size=N)
existing_loans = np.clip(existing_loans, 0, 6)

credit_amount = net_income * np.random.uniform(0.8, 6.5, N) + existing_loans * 500
credit_amount = np.clip(credit_amount, 500, 50000)

debt_to_income = credit_amount / (net_income * 12)
debt_to_income = np.round(debt_to_income, 2)

## Bonitäts- und Verhaltensmerkmale

In [54]:
late_payments = np.random.poisson(lam=1.0, size=N)
late_payments = np.clip(late_payments, 0, 12)

hard_inquiries = np.random.poisson(lam=2.0, size=N)
hard_inquiries = np.clip(hard_inquiries, 0, 10)

credit_history_years = np.random.normal(loc=8, scale=5, size=N)
credit_history_years = np.clip(credit_history_years, 0, 40)

negative_entry = np.random.choice([0, 1], size=N, p=[0.82, 0.18])

collection_cases = np.where(
    negative_entry == 1,
    np.random.poisson(lam=1.5, size=N),
    0
)
collection_cases = np.clip(collection_cases, 0, 5)

bank_balance = np.random.normal(loc=3500, scale=4000, size=N)
bank_balance = np.clip(bank_balance, -2000, 50000)

liquidity_ratio = np.round(bank_balance / monthly_expenses, 2)

## Sonderfall: hohe Liquidität trotz negativem Eintrag

In [55]:
high_income_bad_schufa_idx = np.random.choice(
    np.arange(N),
    size=int(N * 0.05),
    replace=False
)

net_income[high_income_bad_schufa_idx] = np.random.uniform(
    3500, 6000, size=len(high_income_bad_schufa_idx)
)

negative_entry[high_income_bad_schufa_idx] = 1
collection_cases[high_income_bad_schufa_idx] = np.random.randint(
    1, 3, size=len(high_income_bad_schufa_idx)
)

bank_balance[high_income_bad_schufa_idx] = np.random.uniform(
    4000, 15000, size=len(high_income_bad_schufa_idx)
)

liquidity_ratio[high_income_bad_schufa_idx] = np.round(
    bank_balance[high_income_bad_schufa_idx] / monthly_expenses[high_income_bad_schufa_idx], 2
)

## Score-Simulation

In [56]:
old_score = (
    100
    - late_payments * 4
    - hard_inquiries * 2
    - negative_entry * 25
    - collection_cases * 8
    + credit_history_years * 1.5
    - debt_to_income * 10
    + np.random.normal(0, 5, N)
)

old_score = np.clip(old_score, 0, 100)
old_score = np.round(old_score, 1)

In [57]:
new_score = (
    100
    - late_payments * 3
    - hard_inquiries * 1.5
    - negative_entry * 18
    - collection_cases * 6
    + credit_history_years * 1.8
    - debt_to_income * 8
    + liquidity_ratio * 2.5
    + (net_income / 1000) * 1.2
    + np.random.normal(0, 4, N)
)

new_score = np.clip(new_score, 0, 100)
new_score = np.round(new_score, 1)

## Kreditentscheidung

In [58]:
credit_approved_old_system = np.where(
    (old_score >= 60) & (debt_to_income < 0.45),
    1, 0
)

credit_approved_new_system = np.where(
    (new_score >= 60) & (debt_to_income < 0.50),
    1, 0
)

## Default-Simulation

In [59]:
risk_score = (
    -2.2
    + late_payments * 0.18
    + negative_entry * 0.9
    + collection_cases * 0.35
    + np.where(net_income < 1800, 0.8, 0)
    + np.where(debt_to_income > 0.5, 1.1, 0)
    - np.where(bank_balance > 5000, 0.5, 0)
)

default_risk = 1 / (1 + np.exp(-risk_score))
default = np.random.binomial(1, default_risk)

## DataFrame erstellen

In [60]:
df = pd.DataFrame({
    'age': age,
    'net_income': np.round(net_income, 2),
    'employment_status': employment_status,
    'marital_status': marital_status,
    'dependents': dependents,
    'monthly_expenses': np.round(monthly_expenses, 2),
    'existing_loans': existing_loans,
    'credit_amount': np.round(credit_amount, 2),
    'debt_to_income': debt_to_income,
    'late_payments': late_payments,
    'hard_inquiries': hard_inquiries,
    'credit_history_years': np.round(credit_history_years, 1),
    'negative_entry': negative_entry,
    'collection_cases': collection_cases,
    'bank_balance': np.round(bank_balance, 2),
    'liquidity_ratio': liquidity_ratio,
    'old_score': old_score,
    'new_score': new_score,
    'credit_approved_old_system': credit_approved_old_system,
    'credit_approved_new_system': credit_approved_new_system,
    'default': default
})

df.head()

,age,net_income,employment_status,marital_status,dependents,monthly_expenses,existing_loans,credit_amount,debt_to_income,late_payments,...,credit_history_years,negative_entry,collection_cases,bank_balance,liquidity_ratio,old_score,new_score,credit_approved_old_system,credit_approved_new_system,default
0,56,3951.44,selbstständig,geschieden,0,2191.64,3,8568.39,0.18,2,...,0.6,1,1,1560.55,0.71,52.3,77.4,0,1,1
1,69,2978.88,angestellt,verheiratet,3,2307.15,1,7140.92,0.20,0,...,11.6,1,2,6168.70,2.67,75.3,100.0,1,1,0
2,46,4336.99,angestellt,verheiratet,1,2860.34,2,25973.59,0.50,0,...,2.6,0,0,5519.57,1.93,84.2,100.0,0,0,0
3,32,1325.27,teilzeit,ledig,1,1204.22,3,8379.37,0.53,2,...,11.8,1,4,2690.47,2.23,38.2,68.7,0,0,1
4,60,3724.80,angestellt,verheiratet,3,3641.84,3,5408.43,0.12,3,...,0.0,0,0,-2000.00,-0.55,80.7,92.6,1,1,0


## Qualitätschecks und Plausibilitätsprüfung

In [61]:
print("Shape:", df.shape)
print("\nFehlende Werte:")
print(df.isna().sum())

print("\nDuplikate:")
print(df.duplicated().sum())

print("\nDefault-Rate:")
print(df['default'].value_counts(normalize=True))

print("\nKorrelationsmatrix der Scores:")
print(df[['old_score', 'new_score']].corr())

df.describe()

Shape: (10000, 21)

Fehlende Werte:
age                           0
net_income                    0
employment_status             0
marital_status                0
dependents                    0
monthly_expenses              0
existing_loans                0
credit_amount                 0
debt_to_income                0
late_payments                 0
hard_inquiries                0
credit_history_years          0
negative_entry                0
collection_cases              0
bank_balance                  0
liquidity_ratio               0
old_score                     0
new_score                     0
credit_approved_old_system    0
credit_approved_new_system    0
default                       0
dtype: int64

Duplikate:
0

Default-Rate:
default
0    0.8012
1    0.1988
Name: proportion, dtype: float64

Korrelationsmatrix der Scores:
           old_score  new_score
old_score   1.000000   0.790522
new_score   0.790522   1.000000


,age,net_income,dependents,monthly_expenses,existing_loans,credit_amount,debt_to_income,late_payments,hard_inquiries,credit_history_years,negative_entry,collection_cases,bank_balance,liquidity_ratio,old_score,new_score,credit_approved_old_system,credit_approved_new_system,default
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,43.539400,2912.820866,1.058400,1955.490326,1.198000,10776.448538,0.324817,1.004700,2.012800,8.094060,0.220700,0.342300,3956.470657,2.548268,89.101170,96.914840,0.695900,0.860200,0.198800
std,14.911636,1243.729622,1.201388,822.228848,1.090649,6598.808789,0.139160,1.008851,1.431238,4.738524,0.414739,0.825105,3901.181092,3.172712,15.816178,7.498258,0.460048,0.346797,0.399117
min,18.000000,800.000000,0.000000,500.000000,0.000000,668.050000,0.070000,0.000000,0.000000,0.000000,0.000000,0.000000,-2000.000000,-4.000000,6.800000,40.400000,0.000000,0.000000,0.000000
25%,31.000000,1923.947500,0.000000,1334.390000,0.000000,5568.625000,0.200000,0.000000,1.000000,4.600000,0.000000,0.000000,993.370000,0.500000,84.400000,100.000000,0.000000,1.000000,0.000000
50%,43.000000,2894.125000,1.000000,1904.850000,1.000000,9321.785000,0.330000,1.000000,2.000000,7.950000,0.000000,0.000000,3666.400000,1.930000,97.100000,100.000000,1.000000,1.000000,0.000000
75%,56.000000,3791.492500,2.000000,2510.215000,2.000000,14664.077500,0.440000,2.000000,3.000000,11.300000,0.000000,0.000000,6560.155000,3.770000,100.000000,100.000000,1.000000,1.000000,0.000000
max,69.000000,7921.300000,4.000000,6326.420000,6.000000,40917.520000,0.740000,7.000000,9.000000,24.900000,1.000000,5.000000,18694.810000,27.130000,100.000000,100.000000,1.000000,1.000000,1.000000


## Sonderfall-Gruppe analysieren

In [62]:
special_case = df[
    (df['negative_entry'] == 1) &
    (df['net_income'] >= 4000) &
    (df['bank_balance'] >= 4000)
]

special_case[['net_income', 'bank_balance', 'old_score', 'new_score',
              'credit_approved_old_system', 'credit_approved_new_system', 'default']].describe()

,net_income,bank_balance,old_score,new_score,credit_approved_old_system,credit_approved_new_system,default
count,530.000000,530.000000,530.000000,530.000000,530.000000,530.000000,530.000000
mean,4954.599283,8855.744774,63.603396,92.821698,0.467925,0.867925,0.284906
std,605.691709,3116.221141,11.686689,8.860560,0.499441,0.338893,0.451796
min,4009.340000,4001.020000,28.900000,54.300000,0.000000,0.000000,0.000000
25%,4422.942500,6036.867500,55.625000,87.625000,0.000000,1.000000,0.000000
50%,4911.440000,8655.175000,63.600000,96.850000,0.000000,1.000000,0.000000
75%,5453.147500,11423.792500,71.400000,100.000000,1.000000,1.000000,1.000000
max,7142.570000,14988.640000,100.000000,100.000000,1.000000,1.000000,1.000000


## Export als CSV und Download

Die Datei wird in Colab erzeugt und anschließend auf deinen Computer heruntergeladen.
Danach kannst du sie händisch in deinen Projektordner auf dem Schreibtisch verschieben.

In [63]:
from google.colab import files

file_name = "test_credit_upload_file.csv"

df.to_csv(file_name, index=False, encoding="utf-8")

print("Datei gespeichert in Colab unter:")
print("/content/" + file_name)

files.download(file_name)

Datei gespeichert in Colab unter:
/content/test_credit_upload_file.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>